# 第 34 章 · C++ / CUDA 扩展：TORCH_LIBRARY 把 C++ 函数登记进分派表

本章回答：什么时候需要写 C++ / CUDA 扩展而不是 Triton？`TORCH_LIBRARY` 和 `TORCH_LIBRARY_IMPL` 各做什么？`load_inline` 和 `setup.py` 两种构建方式怎么选？为什么手写的 C++ 不一定比 PyTorch 自带的快？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/34-cpp-extension.md`（讲解、动画、练习都在讲义里）。对应原文：官方教程「Custom C++ and CUDA Operators」与 `torch.utils.cpp_extension` 文档（〔在线〕docs.pytorch.org/tutorials/advanced/cpp_custom_ops.html）；本讲义 [第 3 章 分派器](03-dispatcher.md)；CUDA 讲义 [第 25 章 编译部署](../../../cuda-programming/notes/lessons/25-compile-deploy.md)；本章编译和计时是本机 Mac（clang + ninja）torch 2.15.0 实测

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
import os
import sys
import time

from torch.utils.cpp_extension import load_inline
import torch

torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 三段 C++ | TORCH_LIBRARY + IMPL | `build("...", ["-O3"])` | [0, 0, 4]；opcheck 全部 SUCCESS |
| 第三步 编译 | load_inline | `build` 的返回值 | 约 5–7 秒 |
| 第四步 四方对比 | eager / -O0 / -O3 / compile | `per_call_ms(...)` | 约 1.76 / 108 / 1.68 / 1.17 ms |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**常量 `SRC`**（来自 `ch34_cpp_extension.py`）

In [ ]:
SRC = r"""
#include <torch/extension.h>
at::Tensor add_relu_cpu(const at::Tensor& x, const at::Tensor& y) {
  TORCH_CHECK(x.sizes() == y.sizes(), "shape mismatch");
  auto xc = x.contiguous(), yc = y.contiguous();
  auto out = torch::empty_like(xc);
  const float* a = xc.data_ptr<float>(); const float* b = yc.data_ptr<float>(); float* o = out.data_ptr<float>();
  at::parallel_for(0, xc.numel(), 1 << 16, [&](int64_t s, int64_t e) {
    for (int64_t i = s; i < e; ++i) { float v = a[i] + b[i]; o[i] = v > 0 ? v : 0; }
  });
  return out;
}
TORCH_LIBRARY(NS, m) { m.def("add_relu(Tensor x, Tensor y) -> Tensor"); }
TORCH_LIBRARY_IMPL(NS, CPU, m) { m.impl("add_relu", &add_relu_cpu); }
"""

**函数 `build`**（来自 `ch34_cpp_extension.py`）：即时编译并登记 torch.ops.<ns>.add_relu；返回编译秒数。

In [ ]:
def build(ns: str, flags: list):
    """即时编译并登记 torch.ops.<ns>.add_relu；返回编译秒数。"""
    os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]   # ninja 在 .venv/bin
    t = time.time()
    load_inline(name=ns, cpp_sources=SRC.replace("NS", ns), is_python_module=False, extra_cflags=flags)
    torch.library.register_fake(f"{ns}::add_relu")(lambda x, y: torch.empty_like(x))
    return time.time() - t

**函数 `per_call_ms`**（来自 `ch34_cpp_extension.py`）：下面要用到的小工具。

In [ ]:
def per_call_ms(fn, reps: int = 20) -> float:
    fn()
    fn()
    t = time.perf_counter()
    for _ in range(reps):
        fn()
    return (time.perf_counter() - t) / reps * 1e3

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 编译（-O3）用时", round(build("ptdd_ext_o3", ["-O3"]), 1), "s")
op = torch.ops.ptdd_ext_o3.add_relu
print("结果：", op(torch.tensor([-2.0, 1.0, 3.0]), torch.tensor([1.0, -3.0, 1.0])).tolist())
print("opcheck：", torch.library.opcheck(op.default, (torch.randn(5), torch.randn(5))))
build("ptdd_ext_o0", ["-O0"])
a, b = torch.randn(1 << 24), torch.randn(1 << 24)
cf = torch.compile(lambda: torch.relu(a + b))

In [ ]:
print("\n第四步 每次 ms：eager", round(per_call_ms(lambda: torch.relu(a + b)), 2),
      "-O0", round(per_call_ms(lambda: torch.ops.ptdd_ext_o0.add_relu(a, b)), 2),
      "-O3", round(per_call_ms(lambda: op(a, b)), 2), "compile", round(per_call_ms(cf), 2))

## 3. 代码里才看得到的

1. **`build` 把 .venv/bin 加进 PATH**：讲义第三步说的 ninja 找不到的坑，代码里直接绕过去了。
2. **-O0 比 -O3 慢约 60 倍**：测试只核对"慢 10 倍以上"。

## 4. 自己改一改

- 把 `parallel_for` 的分块大小从 1 << 16 改成 1 << 10，看 -O3 版变快还是变慢。
- 把 C++ 里的 `x.contiguous()` 去掉，传一个转置过的输入，看结果对不对。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 先 compile、再 Triton，要 CUTLASS、PTX、包装库才写 C++；
> 实现 + TORCH_LIBRARY 加一行 + TORCH_LIBRARY_IMPL 按设备填一列，fake 和反向 Python 里补；
> load_inline 学习、setup.py 发布，ninja 要在 PATH；-O3 先写对，手写不自动赢，三方实测再定。